In [1]:
import numpy as np
import pandas as pd


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [33]:
def webscrape_votes(num):
    # ── CONFIG ────────────────────────────────────────────────────────────────────
    CONGRESS = num
    OUT_FILE = f"senate_votes_{num}.csv"

    MEMBERS_URL   = f"https://voteview.com/static/data/out/members/S{CONGRESS:03d}_members.csv"
    VOTES_URL     = f"https://voteview.com/static/data/out/votes/S{CONGRESS:03d}_votes.csv"
    ROLLCALLS_URL = f"https://voteview.com/static/data/out/rollcalls/S{CONGRESS:03d}_rollcalls.csv"

    # SSL workaround: disable verification
    session = requests.Session()
    session.verify = False
    requests.packages.urllib3.disable_warnings()  # suppress the InsecureRequestWarning

    def fetch_csv(url, retries=5, backoff=10):
        for attempt in range(retries):
            try:
                resp = session.get(url, timeout=30)
                resp.raise_for_status()
                return pd.read_csv(io.StringIO(resp.text))
            except Exception as e:
                if attempt < retries - 1:
                    wait = backoff * (attempt + 1)
                    print(f"  ⚠️  Error ({e.__class__.__name__}), retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    raise

    print("Downloading members...")
    members = fetch_csv(MEMBERS_URL)
    members = members[["icpsr", "bioname", "bioguide_id", "party_code", "state_abbrev"]]
    party_labels = {100: "Democrat", 200: "Republican", 328: "Independent"}
    members["party"] = members["party_code"].map(party_labels).fillna("Other")
    print(f"  {len(members)} senators found")

    print("Downloading roll call metadata...")
    rollcalls = fetch_csv(ROLLCALLS_URL)
    rollcalls = rollcalls[["rollnumber", "bill_number", "vote_desc", "vote_question"]]
    print(f"  {len(rollcalls)} roll calls found")

    print("Downloading votes...")
    votes = fetch_csv(VOTES_URL)
    print(f"  {len(votes)} vote records found")

    # Map cast codes to readable labels
    def cast_label(code):
        if code in (1, 2, 3): return "Yea"
        if code in (4, 5, 6): return "Nay"
        if code in (7, 8, 9): return "Not Voting"
        return None

    votes["vote"] = votes["cast_code"].map(cast_label)
    votes = votes.dropna(subset=["vote"])

    # Join everything
    df = (
        votes
        .merge(members, on="icpsr", how="left")
        .merge(rollcalls, on="rollnumber", how="left")
    )[["bioname", "party", "state_abbrev", "icpsr", "bioguide_id",
    "bill_number", "vote_desc", "vote_question", "vote", "rollnumber"]]

    df.columns = ["senator", "party", "state", "icpsr", "bioguide_id",
                "bill_id", "bill_desc", "vote_question", "vote", "rollnumber"]

    df.to_csv(OUT_FILE, index=False)
    print(f"\n✅ Done! {len(df)} rows, {df['bill_id'].nunique()} bills, {df['senator'].nunique()} senators")
    print(f"Saved to {OUT_FILE}")
    print(df.head(10).to_string(index=False))

In [34]:
for i in range (101,119):
    webscrape_votes(i)

  103 senators found
  638 roll calls found
  64403 vote records found

✅ Done! 64403 rows, 172 bills, 102 senators
Saved to senate_votes_101.csv
                    senator      party state  icpsr bioguide_id bill_id                                                    bill_desc     vote_question vote  rollnumber
BENTSEN, Lloyd Millard, Jr.   Democrat    TX    660     B000401   PN128 James Addison Baker, III, of Texas, to be Secretary of State On the Nomination  Yea           1
  BURDICK, Quentin Northrup   Democrat    ND   1252     B001077   PN128 James Addison Baker, III, of Texas, to be Secretary of State On the Nomination  Yea           1
       BYRD, Robert Carlyle   Democrat    WV   1366     B001210   PN128 James Addison Baker, III, of Texas, to be Secretary of State On the Nomination  Yea           1
         INOUYE, Daniel Ken   Democrat    HI   4812     I000025   PN128 James Addison Baker, III, of Texas, to be Secretary of State On the Nomination  Yea           1
      THURMOND

In [35]:
df101 = pd.read_csv("senate_votes_101.csv")
df102 = pd.read_csv("senate_votes_102.csv")
df103 = pd.read_csv("senate_votes_103.csv")
df104 = pd.read_csv("senate_votes_104.csv")
df105 = pd.read_csv("senate_votes_105.csv")
df106 = pd.read_csv("senate_votes_106.csv")
df107 = pd.read_csv("senate_votes_107.csv")
df108 = pd.read_csv("senate_votes_108.csv")
df109 = pd.read_csv("senate_votes_109.csv")
df110 = pd.read_csv("senate_votes_110.csv")
df111 = pd.read_csv("senate_votes_111.csv")
df112 = pd.read_csv("senate_votes_112.csv")
df113 = pd.read_csv("senate_votes_113.csv")
df114 = pd.read_csv("senate_votes_114.csv")
df115 = pd.read_csv("senate_votes_115.csv")
df116 = pd.read_csv("senate_votes_116.csv")
df117 = pd.read_csv("senate_votes_117.csv")
df118 = pd.read_csv("senate_votes_118.csv")
df119 = pd.read_csv("senate_votes_119.csv")

mega_df = pd.concat([df101,df102,df103,df104,df105,df106,df107,df108,df109,df110,df111,df112,df113,df114,
                     df115,df116,df117,df118,df119])

In [36]:
mega_df=mega_df[["bioguide_id","bill_id","bill_desc","vote"]]

In [37]:
mega_df.head()

,bioguide_id,bill_id,bill_desc,vote
0,B000401,PN128,"James Addison Baker, III, of Texas, to be Secr...",Yea
1,B001077,PN128,"James Addison Baker, III, of Texas, to be Secr...",Yea
2,B001210,PN128,"James Addison Baker, III, of Texas, to be Secr...",Yea
3,I000025,PN128,"James Addison Baker, III, of Texas, to be Secr...",Yea
4,T000254,PN128,"James Addison Baker, III, of Texas, to be Secr...",Yea


In [38]:
mega_df.to_csv('all_votes.csv', index=False)

In [2]:
import pandas as pd

df = pd.read_csv("all_votes_trimmed.csv")

# Remove PN, treaty, SRES, and HRES bills
prefixes_to_remove = ["PN", "SRES", "HRES"]

mask = (
    df["bill_id"].str.contains("treaty", case=False) |
    df["bill_id"].str.extract(r"^([A-Za-z]+)")[0].isin(prefixes_to_remove)
)

df = df[~mask]

df.to_csv("all_votes_trimmed.csv", index=False)
print(f"Remaining rows: {len(df)}")
print(df["bill_id"].str.extract(r"^([A-Za-z]+)")[0].value_counts())

Remaining rows: 925283
0
S          392175
HR         377793
SCONRES     86391
SJRES       26736
HJRES       23873
HCONRES     11767
Name: count, dtype: int64


In [3]:
df = pd.read_csv("all_votes_trimmed.csv")

In [4]:
df.to_parquet("all_votes.parquet", index=False)